# 04 — Clustering Experiments (db-robust-clust)

This notebook contains all clustering experiments that use the professor's
`db-robust-clust` package.  The merged per-image PCA matrix produced by
`06_pca_per_image_merged_clustering.ipynb` serves as the image feature input.

## Step 1 — Install dependencies

In [19]:
import subprocess, sys

def pip_install(*packages):
    for pkg in packages:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "openpyxl", pkg],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"  OK    {pkg}")
        else:
            print(f"  FAIL  {pkg}")
            print(result.stderr[-600:] if result.stderr else "(no stderr)")

# db-robust-clust requires scikit-learn-extra which needs MSVC C++ Build Tools
# to compile on Windows — skipping it.  For 416 patients we call kmedoids.fasterpam
# directly (same algorithm, no subsampling wrapper needed).
pip_install("kmedoids", "robust-mixed-dist")
print("Done.")

  OK    kmedoids
  OK    robust-mixed-dist
Done.


## Step 2 — Imports

In [20]:
import os
import numpy as np
import pandas as pd

# k-medoids solver — fasterpam accepts a precomputed distance matrix
import kmedoids

# Distance functions from the professor's mixed-distance package
from robust_mixed_dist.mixed import (
    generalized_gower_dist_matrix,
    robust_mahalanobis_dist_matrix,
    simple_gower_dist_matrix,
    S_robust,
)

# ── helper: run FasterPAM and return integer labels ─────────────────────────
def run_kmedoids(D, k, random_state=42):
    """Cluster using FasterPAM on a precomputed distance matrix D."""
    result = kmedoids.fasterpam(D, medoids=k, random_state=random_state)
    return np.array(result.labels, dtype=int)

# ── helper: crosstab cluster labels vs CDR ──────────────────────────────────
def show_crosstab(labels, cdr_series, title=""):
    df_tmp = pd.DataFrame({"cluster": labels, "CDR": cdr_series.values})
    df_tmp = df_tmp[df_tmp["CDR"].notnull()].copy()
    df_tmp["CDR"] = df_tmp["CDR"].astype(str)
    if title:
        print(f"\n{'─'*55}")
        print(f"  {title}")
        print(f"{'─'*55}")
    ct = pd.crosstab(df_tmp["cluster"], df_tmp["CDR"],
                     margins=True, margins_name="Total")
    print(ct.to_string())
    print("\nRow % (CDR distribution within each cluster):")
    pct = pd.crosstab(df_tmp["cluster"], df_tmp["CDR"],
                      normalize="index").mul(100).round(1)
    print(pct.to_string())

print("All imports successful.")

All imports successful.


## Step 3 — Load data

In [21]:
DATA_DIR = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
NB_DIR   = os.path.join(DATA_DIR, "notebooks")

# ── merged per-image PCA features (output of notebook 06) ──────────────────
merged_path = os.path.join(NB_DIR, "pca_all_images_merged.csv")
df_pca = pd.read_csv(merged_path)
print(f"PCA merged matrix : {df_pca.shape}")
print(f"  columns preview : {list(df_pca.columns[:5])} ... {list(df_pca.columns[-3:])}")

# ── clinical data (CDR + demographics) ─────────────────────────────────────
clinical_path = os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
df_clinical = pd.read_excel(clinical_path)

# Standardise patient_id column name (OASIS uses 'ID' in the spreadsheet)
if "ID" in df_clinical.columns:
    df_clinical = df_clinical.rename(columns={"ID": "patient_id"})

print(f"\nClinical table    : {df_clinical.shape}")
print(f"  columns         : {list(df_clinical.columns)}")

# ── quick sanity checks ─────────────────────────────────────────────────────
assert "patient_id" in df_pca.columns,      "patient_id missing from PCA file"
assert "patient_id" in df_clinical.columns, "patient_id missing from clinical file"
assert df_pca.shape[0] > 0,                 "PCA file is empty"
print("\nSanity checks passed.")

PCA merged matrix : (416, 551)
  columns preview : ['patient_id', 'cor_PC1', 'cor_PC2', 'cor_PC3', 'cor_PC4'] ... ['sbj_sag_PC108', 'sbj_sag_PC109', 'sbj_sag_PC110']

Clinical table    : (436, 12)
  columns         : ['patient_id', 'M/F', 'Hand', 'Age', 'Educ', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF', 'Delay']

Sanity checks passed.


## Planned Experiments

The following experiments will be implemented in subsequent cells of this
notebook, following the professor's methodology.

---

### Experiment 1 — PCA per-image merged → Robust Mahalanobis

- **Features**: the merged matrix from notebook 06  
  (416 patients × 550 PCA components, 5 image types × 110 components each)
- **Distance**: Robust Mahalanobis via `robust_mahalanobis_dist_matrix`  
  (uses a robust covariance estimator `S_robust` to handle outliers)
- **Clusterer**: `SampleDistClustering` wrapping `kmedoids.fasterpam`
- **Goal**: test whether imaging features alone recover CDR-aligned groups

---

### Experiment 2 — Clinical only → Generalised Gower

- **Features**: clinical variables only — Age, Educ, SES, MMSE, eTIV, nWBV, ASF  
  (mix of quantitative and categorical columns requires a mixed-type distance)
- **Distance**: Generalised Gower via `generalized_gower_dist_matrix`  
  (handles quantitative, binary, and multi-class columns natively)
- **Clusterer**: `SampleDistClustering` wrapping `kmedoids.fasterpam`
- **Goal**: establish a clinical-data-only baseline for comparison

---

### Experiment 3 — Combined clinical + PCA → Generalised Gower

- **Features**: clinical variables joined with the merged PCA matrix  
  (quantitative PCA components + mixed clinical columns)
- **Distance**: Generalised Gower via `generalized_gower_dist_matrix`  
  (all PCA columns treated as quantitative; clinical columns typed accordingly)
- **Clusterer**: `SampleDistClustering` wrapping `kmedoids.fasterpam`
- **Goal**: test whether combining imaging and clinical information  
  yields better CDR-aligned clusters than either source alone

---

All experiments evaluate cluster quality by crosstabulating cluster labels
against CDR (Clinical Dementia Rating) — a held-out ground truth.